This project combines Natural Language Processing (NLP) and Deep Learning to analyze and generate lyrics inspired by Portishead’s music. Using TensorFlow and Keras, an LSTM neural network was trained on a collection of Portishead lyrics to predict text sequences based on emotional tone and lyrical style. Additionally, TextBlob was employed to conduct sentiment analysis, and Plotly visualizations were created to illustrate relationships between sentiment polarity and word count across tracks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from wordcloud import WordCloud, STOPWORDS
import tensorflow as tf
import warnings
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import re
from collections import Counter
import random
from IPython.display import IFrame
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("/kaggle/input/portishead-songs-dataset/portishead_lyrics.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df['cleaned_lyrics'] = df['lyrics_text'].apply(lambda x: re.sub(r'[^\w\s]', '', str(x).lower()).strip())
df.head()

In [ ]:
lyrics_text = " ".join(df["cleaned_lyrics"].dropna())
wordcloud = WordCloud(stopwords=STOPWORDS, background_color="black", colormap="plasma").generate(lyrics_text)

plt.figure(figsize=(12, 8))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Most Common Words in Portishead’s Lyrics")
plt.show()

In [ ]:
from textblob import TextBlob
df["sentiment"] = df["cleaned_lyrics"].apply(lambda x: TextBlob(x).sentiment.polarity)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['sentiment'], bins=20, kde=True, color="skyblue", edgecolor="black")
plt.title("Sentiment Polarity Distribution of Portishead's Lyrics")
plt.xlabel("Sentiment Polarity")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Calculate word count for each song
df['word_count'] = df['cleaned_lyrics'].apply(lambda x: len(x.split()))

# Create the interactive scatter plot with Plotly
fig = px.scatter(
    df,
    x="sentiment",
    y="word_count",
    color="track_name",
    hover_name="track_name",
    title="Sentiment vs Word Count in Portishead's Lyrics",
    labels={"sentiment": "Sentiment", "word_count": "Word Count"},
    template="plotly_dark"
)

# Save the plot as an HTML file
fig.write_html("sentiment_word_count.html")

# Display the HTML file as an interactive iframe in the notebook
IFrame("sentiment_word_count.html", width=850, height=600)

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['cleaned_lyrics'])
total_words = len(tokenizer.word_index) + 1

In [ ]:
input_sequences = []
for line in df['cleaned_lyrics'].dropna():
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [ ]:
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
predictors, labels = input_sequences[:,:-1], input_sequences[:,-1]
labels = tf.keras.utils.to_categorical(labels, num_classes=total_words)


In [ ]:
model = Sequential([
    Embedding(total_words, 64),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(predictors, labels, epochs=50, verbose=1)

In [ ]:
def generate_lyrics(seed_text, next_words=50):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = model.predict(token_list, verbose=0)
        predicted_word = tokenizer.index_word[np.argmax(predicted)]
        seed_text += " " + predicted_word
    return seed_text

In [ ]:
seed_text = random.choice(df['cleaned_lyrics'].dropna().values)
print("Seed Text:", seed_text)
print("Generated Lyrics:\n", generate_lyrics(seed_text))